# CV Transfer Learning - Colab Free

Chọn một free runtime. Notebook chạy thật; cpu-mini dùng FakeData, gpu-free dùng CIFAR10 subset.

## Environment check

In [ ]:
import importlib.util
import platform
HAS_TORCH = importlib.util.find_spec('torch') is not None
assert HAS_TORCH, 'Install the pinned torch/torchvision pair for this runtime'
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PROFILE = 'gpu-free' if DEVICE == 'cuda' else 'cpu-mini'
print({'python': platform.python_version(), 'torch': torch.__version__, 'device': DEVICE, 'profile': PROFILE})

## Install/verify dependencies

Nếu import lỗi, cài torch/torchvision theo selector chính thức của PyTorch rồi restart runtime. Không cài lại khi runtime đã có bản tương thích.

## Configuration and seed

In [ ]:
import random
import numpy as np
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MAX_EPOCHS=5 if PROFILE=='gpu-free' else 1
SAMPLES=3000 if PROFILE=='gpu-free' else 160
BATCH_SIZE=32 if PROFILE=='gpu-free' else 8
IMAGE_SIZE=160 if PROFILE=='gpu-free' else 96
print({'seed':SEED,'epochs':MAX_EPOCHS,'samples':SAMPLES,'batch':BATCH_SIZE})

## Data acquisition with mini fallback

In [ ]:
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights
WEIGHTS=ResNet18_Weights.DEFAULT
mean,std=WEIGHTS.transforms().mean,WEIGHTS.transforms().std
transform=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),transforms.Normalize(mean,std)])
DATA_SOURCE='FakeData cpu-mini fallback'
if PROFILE=='cpu-mini':
    dataset=datasets.FakeData(size=SAMPLES,image_size=(3,IMAGE_SIZE,IMAGE_SIZE),num_classes=2,transform=transform,random_offset=SEED)
    class_names=['class-0','class-1']
else:
    try:
        full=datasets.CIFAR10(root='data',train=True,download=True,transform=transform)
        dataset=Subset(full,list(range(min(SAMPLES,len(full)))))
        class_names=full.classes
        DATA_SOURCE='CIFAR10 subset'
    except Exception as error:
        print(f'CIFAR10 unavailable ({type(error).__name__}); switching to FakeData smoke test. Do not report its accuracy as model quality.')
        PROFILE='cpu-mini-offline-fallback'
        MAX_EPOCHS=1; SAMPLES=160; BATCH_SIZE=8; IMAGE_SIZE=96
        transform=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),transforms.Normalize(mean,std)])
        dataset=datasets.FakeData(size=SAMPLES,image_size=(3,IMAGE_SIZE,IMAGE_SIZE),num_classes=2,transform=transform,random_offset=SEED)
        class_names=['class-0','class-1']
        DATA_SOURCE='FakeData internet fallback'
generator=torch.Generator().manual_seed(SEED)
train_n=int(0.7*len(dataset)); val_n=int(0.15*len(dataset)); test_n=len(dataset)-train_n-val_n
train_ds,val_ds,test_ds=random_split(dataset,[train_n,val_n,test_n],generator=generator)
loaders={name:DataLoader(ds,batch_size=BATCH_SIZE,shuffle=name=='train',num_workers=0) for name,ds in [('train',train_ds),('validation',val_ds),('test',test_ds)]}
print({'source':DATA_SOURCE,'normalization':{'mean':mean,'std':std},**{k:len(v.dataset) for k,v in loaders.items()}})


## Data validation

In [ ]:
assert min(len(loader.dataset) for loader in loaders.values())>0
images,labels=next(iter(loaders['train']))
assert images.ndim==4 and labels.ndim==1
NUM_CLASSES=len(class_names)
print({'shape':tuple(images.shape),'classes':NUM_CLASSES})

## Baseline

In [ ]:
from collections import Counter
train_labels=[]
for _,labels in loaders['train']: train_labels.extend(labels.tolist())
majority=max(Counter(train_labels).values())/len(train_labels)
print({'majority_accuracy':majority})

## Training

In [ ]:
from pathlib import Path
from torchvision.models import resnet18
from torch import nn
weights=None if PROFILE.startswith('cpu-mini') else WEIGHTS
try:
    model=resnet18(weights=weights)
except Exception as error:
    print(f'Pretrained weights unavailable ({type(error).__name__}); using random weights for execution-only fallback.')
    PROFILE='cpu-mini-offline-fallback'; DATA_SOURCE='FakeData or cached data; random-weight fallback'
    model=resnet18(weights=None)
for parameter in model.parameters(): parameter.requires_grad=False
model.fc=nn.Linear(model.fc.in_features,NUM_CLASSES)
model=model.to(DEVICE)
optimizer=torch.optim.Adam(model.fc.parameters(),lr=1e-3)
criterion=nn.CrossEntropyLoss(); best=float('inf'); patience=2; stale=0; history=[]
Path('artifacts').mkdir(exist_ok=True)
checkpoint_path=Path('artifacts/checkpoint.pt')
start_epoch=0
RESUME=False  # Set True after an interrupted run; keep the same config and label mapping.
if RESUME and checkpoint_path.exists():
    checkpoint=torch.load(checkpoint_path,map_location=DEVICE,weights_only=True)
    model.load_state_dict(checkpoint['model']); optimizer.load_state_dict(checkpoint['optimizer'])
    start_epoch=int(checkpoint['epoch']); best=float(checkpoint['best_validation_loss'])
    history=list(checkpoint['history'])
    assert checkpoint['config']=={'profile':PROFILE,'samples':SAMPLES,'batch_size':BATCH_SIZE,'image_size':IMAGE_SIZE,'max_epochs':MAX_EPOCHS}
    assert checkpoint['class_names']==class_names
for epoch in range(start_epoch,MAX_EPOCHS):
    model.train(); train_loss=0.0
    for x,y in loaders['train']:
        x,y=x.to(DEVICE),y.to(DEVICE); optimizer.zero_grad(); loss=criterion(model(x),y); loss.backward(); optimizer.step(); train_loss+=loss.item()*len(x)
    model.eval(); val_loss=0.0
    with torch.no_grad():
        for x,y in loaders['validation']:
            x,y=x.to(DEVICE),y.to(DEVICE); val_loss+=criterion(model(x),y).item()*len(x)
    row={'epoch':epoch+1,'train_loss':train_loss/len(train_ds),'validation_loss':val_loss/len(val_ds)}; history.append(row); print(row)
    if row['validation_loss']<best:
        best=row['validation_loss']; stale=0
        torch.save({'model':model.state_dict(),'optimizer':optimizer.state_dict(),'epoch':epoch+1,'best_validation_loss':best,'history':history,'seed':SEED,'profile':PROFILE,'config':{'profile':PROFILE,'samples':SAMPLES,'batch_size':BATCH_SIZE,'image_size':IMAGE_SIZE,'max_epochs':MAX_EPOCHS},'class_names':class_names},checkpoint_path)
    else:
        stale+=1
        if stale>=patience: break
# Optional stretch: unfreeze layer4 after the frozen-head baseline, then train one controlled extra epoch.
UNFREEZE_LAST_BLOCK=False
fine_tune_history=[]
if UNFREEZE_LAST_BLOCK and not PROFILE.startswith('cpu-mini'):
    for parameter in model.layer4.parameters(): parameter.requires_grad=True
    optimizer=torch.optim.Adam([{'params':model.layer4.parameters(),'lr':1e-5},{'params':model.fc.parameters(),'lr':1e-4}])
    model.train(); fine_tune_loss=0.0
    for x,y in loaders['train']:
        x,y=x.to(DEVICE),y.to(DEVICE); optimizer.zero_grad(); loss=criterion(model(x),y); loss.backward(); optimizer.step(); fine_tune_loss+=loss.item()*len(x)
    model.eval(); fine_tune_validation_loss=0.0
    with torch.no_grad():
        for x,y in loaders['validation']:
            x,y=x.to(DEVICE),y.to(DEVICE); fine_tune_validation_loss+=criterion(model(x),y).item()*len(x)
    fine_tune_history.append({'policy':'unfreeze-layer4','train_loss':fine_tune_loss/len(train_ds),'validation_loss':fine_tune_validation_loss/len(val_ds)})
    print({'frozen_head_best_validation_loss':best,'fine_tune':fine_tune_history[-1]})


## Evaluation and error analysis

In [ ]:
from PIL import Image,ImageDraw
model.load_state_dict(torch.load('artifacts/checkpoint.pt',map_location=DEVICE,weights_only=True)['model']); model.eval()
truth=[];predicted=[];failures=[]
failure_dir=Path('artifacts/failure-images'); failure_dir.mkdir(parents=True,exist_ok=True)
def save_failure_image(tensor,path,expected,actual):
    mean_tensor=torch.tensor(mean).view(3,1,1); std_tensor=torch.tensor(std).view(3,1,1)
    pixels=(tensor.cpu()*std_tensor+mean_tensor).clamp(0,1)
    image=transforms.ToPILImage()(pixels)
    canvas=Image.new('RGB',(image.width,image.height+24),'white'); canvas.paste(image,(0,24))
    ImageDraw.Draw(canvas).text((4,4),f'truth={class_names[expected]} predicted={class_names[actual]}',fill='black')
    canvas.save(path)
with torch.no_grad():
    for batch,(x,y) in enumerate(loaders['test']):
        logits=model(x.to(DEVICE)); probabilities=torch.softmax(logits,dim=1).cpu(); p=probabilities.argmax(1)
        truth.extend(y.tolist()); predicted.extend(p.tolist())
        for i,(expected,actual) in enumerate(zip(y.tolist(),p.tolist())):
            if expected!=actual and len(failures)<20:
                item_id=f'batch-{batch}-item-{i}'; image_path=f'artifacts/failure-images/{item_id}.png'
                save_failure_image(x[i],image_path,expected,actual)
                failures.append({'id':item_id,'truth':expected,'predicted':actual,'confidence':float(probabilities[i,actual]),'image':image_path,'reason':'manual-review-required'})
from sklearn.metrics import accuracy_score,confusion_matrix,f1_score,precision_recall_fscore_support
precision,recall,f1,support=precision_recall_fscore_support(truth,predicted,labels=list(range(NUM_CLASSES)),zero_division=0)
metrics={'accuracy':accuracy_score(truth,predicted),'macro_f1':f1_score(truth,predicted,average='macro'),'per_class':{str(i):{'precision':float(precision[i]),'recall':float(recall[i]),'f1':float(f1[i]),'support':int(support[i])} for i in range(NUM_CLASSES)},'confusion_matrix':confusion_matrix(truth,predicted,labels=list(range(NUM_CLASSES))).tolist(),'failure_examples':failures}
assert len(list(failure_dir.glob('*.png')))==len(failures)<=20
failure_limitation=None if len(failures)==20 else f'Only {len(failures)} misclassifications existed; exported all instead of padding to 20.'
metrics['failure_evidence']={'exported':len(failures),'cap':20,'limitation':failure_limitation}
print({'metrics':metrics,'failure_images':len(failures),'failure_dir':str(failure_dir),'limitation':failure_limitation})


## Save artifacts and manifest

In [ ]:
import hashlib
import json
import shutil
from pathlib import Path
Path('artifacts/metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
Path('artifacts/history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
checksum=hashlib.sha256(Path('artifacts/checkpoint.pt').read_bytes()).hexdigest()
manifest={'seed':SEED,'profile':PROFILE,'device':DEVICE,'data_source':DATA_SOURCE,'quality_evidence':DATA_SOURCE=='CIFAR10 subset','epochs_completed':len(history),'checkpoint_sha256':checksum,'classes':class_names}
Path('artifacts/manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
Path('artifacts/model-card.md').write_text('# Model card\n\nFrozen ResNet18 learning lab. Not for production. FakeData runs validate execution only; they are not model-quality evidence.\n',encoding='utf-8')
archive=shutil.make_archive('artifacts','zip',root_dir='artifacts')
assert Path(archive).is_file()
print({'download':archive,'bytes':Path(archive).stat().st_size,'files':sorted(p.name for p in Path('artifacts').iterdir())})


## Release runtime

Tải `artifacts.zip`. Colab: Runtime > Disconnect and delete runtime. Kaggle: Save Version, download output, tắt accelerator/session.